In [ ]:
import numpy as np
import gudhi as gd
import matplotlib.pyplot as plt
import networkx as nx
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

# Generate a symmetric distance matrix
#np.random.seed(42)  # For reproducibility
#n = 15  # Number of proteins
#random_distances = np.random.uniform(0.1, 1.0, size=(n, n))
#distance_matrix = (random_distances + random_distances.T) / 2  # Symmetrize
#np.fill_diagonal(distance_matrix, 0)  # Set diagonal to 0

def load_distance_matrix(file_name):
    return pd.read_excel(file_name, index_col=0).values

# Function to compute Rips complex from a distance matrix
def compute_rips_from_distance_matrix(distance_matrix, max_edge_length):
    rips_complex = gd.RipsComplex(distance_matrix=distance_matrix, max_edge_length=max_edge_length)
    simplex_tree = rips_complex.create_simplex_tree(max_dimension=2)  # Capture up to 2D simplices
    return simplex_tree

# Function to compute persistent homology and track hole formation
def compute_persistence_1D(simplex_tree):
    persistence = simplex_tree.persistence()
    H1_intervals = []
    hole_data = []

    # Sort edges by filtration value
    edges = []
    faces = []
    for simplex, filtration_value in simplex_tree.get_filtration():
        if len(simplex) == 2:  # Edges
            edges.append((simplex[0], simplex[1], filtration_value))
        elif len(simplex) == 3:  # Triangles (2-simplices)
            faces.append((simplex[0], simplex[1], simplex[2], filtration_value))

    edges.sort(key=lambda x: x[2])  # Sort edges by filtration value (birth time)

    # Dynamic graph construction to track cycle formation
    G = nx.Graph()
    for i, interval in enumerate(persistence):
        if interval[0] == 1:  # H1 homology (1D holes)
            birth, death = interval[1]
            H1_intervals.append((birth, death))

            # Add edges progressively until reaching birth time
            G.clear()
            for u, v, filt in edges:
                if filt > birth:
                    break
                G.add_edge(u, v)

            # Detect the first cycle that forms
            cycles = list(nx.cycle_basis(G))
            participating_vertices = set()
            for cycle in cycles:
                if len(cycle) > 2:  # Ensure it's a valid hole (not a trivial triangle)
                    participating_vertices.update(cycle)

            hole_data.append({
                "hole_index": i + 1,
                "birth": birth,
                "death": death,
                "vertices": sorted(participating_vertices)
            })

    return H1_intervals, hole_data, faces

# Function to visualize the Rips complex and display persistence barcodes for H1 (1D holes)
def visualize_rips_complex_and_h1_persistence(radius):
    simplex_tree = compute_rips_from_distance_matrix(distance_matrix, radius)
    H1_intervals, hole_data, faces = compute_persistence_1D(simplex_tree)

    G = nx.Graph()
    for simplex, filtration in simplex_tree.get_filtration():
        if len(simplex) == 2:
            G.add_edge(simplex[0], simplex[1])

    pos = nx.spring_layout(G, seed=42)
    labels = {i: f"Protein {i+1}" for i in G.nodes()}  # Label nodes as Protein 1, 2, 3, etc.

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # ---- Rips Complex Visualization ----
    axes[0].set_title(f"Vietoris-Rips Complex (Max Edge = {radius:.2f})")
    nx.draw_networkx_nodes(G, pos, ax=axes[0], node_size=500, node_color="red", alpha=0.8)
    nx.draw_networkx_edges(G, pos, ax=axes[0], width=2, edge_color="blue")
    nx.draw_networkx_labels(G, pos, labels, font_size=8, font_color="black", font_weight="bold", ax=axes[0])

    # Draw filled triangles for 2-simplices
    for face in faces:
        if face[3] <= radius:  # Only show triangles that exist at this filtration step
            pts = np.array([pos[face[0]], pos[face[1]], pos[face[2]]])
            axes[0].fill(pts[:, 0], pts[:, 1], color='cyan', alpha=0.3)

    axes[0].axis("off")

    # ---- Persistence Barcode for H1 (1D Holes) ----
    axes[1].set_title("Persistence Barcode ($H_1$ - 1D Holes)")
    y_positions = list(range(len(H1_intervals)))

    for i, (birth, death) in enumerate(H1_intervals):
        death_value = death if death != float("inf") else 1.2  # Cap infinite lifespans
        axes[1].plot([birth, death_value], [y_positions[i], y_positions[i]], 'b-', lw=2)
        axes[1].scatter(birth, y_positions[i], color='red', zorder=3)
        if death != float("inf"):
            axes[1].scatter(death, y_positions[i], color='black', zorder=3)

    axes[1].set_xlabel("Filtration Value")
    axes[1].set_ylabel("Hole Index")
    axes[1].set_yticks(y_positions)
    axes[1].grid(True)

    plt.show()

    # Print detected 1D holes and their participating vertices
    print(f"\nNumber of 1D holes detected: {len(hole_data)}")
    print("Persistence Intervals and Participating Vertices for 1D Holes:")
    for hole in hole_data:
        birth = hole["birth"]
        death = hole["death"] if hole["death"] != float("inf") else "infinity"
        vertices = ", ".join([f"Protein {v+1}" for v in hole["vertices"]])
        print(f"Hole {hole['hole_index']}: Birth = {birth:.2f}, Death = {death}, Vertices: {vertices}")

# Interactive slider for choosing max edge length (filtration threshold)
if __name__ == "__main__":
    distance_matrix_file = "trial_distance_matrix.xlsx"
    distance_matrix = load_distance_matrix(distance_matrix_file)
    n = distance_matrix.shape[0]

    radius_slider = widgets.FloatSlider(value=0.2, min=0.0, max=np.max(distance_matrix), step=0.01, description='Max Edge Length:')
    interactive_plot = widgets.interactive(visualize_rips_complex_and_h1_persistence, radius=radius_slider)

    display(interactive_plot)


interactive(children=(FloatSlider(value=0.2, description='Max Edge Length:', max=1.708115479295449, step=0.01)…